# TinyIDS: how the deployed model (`top14`) is trained

This notebook builds the model that runs on the Raspberry Pi: **`pi/models/top14/model.txt`**.
It is the same code as `training/train.py` (that script also trains the 5 / 7 / 10-feature versions; this notebook
only does the deployed 14-feature one, step by step, with explanations).

**What the model does:** it looks at 10 packets at a time ("one window") and answers one of 3 classes:
**Benign** (normal), **Flood** (DoS / DDoS flood), **Recon** (scanning, e.g. nmap).

**Steps**
1. Settings: which features, which attacks, which classes
2. Load the dataset rows (CICIoT2023) we keep
3. Load our own recorded rows (Docker testbed + real Pi lab)
4. Build train / validation / test sets
5. Train LightGBM on the 14 features and rank them by SHAP
6. Train the final model with the features in ranked order (= the deployed model)
7. Score it on the test rows
8. Save the model and check it is identical to the one on the Pi
9. Save the exact rows it was trained on into `evaluation_matrices/dataset/`

**Run it from the `retrain/` folder** (the folder this notebook is in). It needs the dataset converted to parquet in
`../project/model/cache/` (or `../model/cache/`). Takes ~1-2 minutes.

In [ ]:
import json
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score

HERE = Path.cwd()                                         # = retrain/
CACHE = HERE.parent / "model" / "cache"                   # the CICIoT2023 dataset as parquet (read only)
if not CACHE.exists():
    CACHE = HERE.parent / "project" / "model" / "cache"
OUR_DATA = HERE / "training" / "data"                     # our own recorded rows (csv)
OUT = HERE / "notebook_output" / "top14"                  # where THIS notebook saves its model (does not touch pi/)
DEPLOYED = HERE / "pi" / "models" / "top14"               # the model that runs on the Pi
DATASET_OUT = HERE / "evaluation_matrices" / "dataset"    # the exact training rows are exported here
print("dataset cache:", CACHE, CACHE.exists())

## 1. Settings

**The 14 features.** CICIoT2023 has 46 features per row. We use only the 14 that our Pi computes **exactly the same
way** as the dataset (measured 16 Nov, `dataset.md` section 3): packet sizes, TCP flags and protocol shares.
The other 32 are computed differently live, are shortcuts (e.g. `IAT` = capture time), or are always 0 for our
attacks. If the model learned from a feature the Pi computes differently, it would work in training and fail live.

**The 3 classes.** All 9 flood types (SYN, UDP, ICMP, TCP, PSH-ACK, RST-FIN; DoS and DDoS) become one class
**Flood**; 3 scan types become **Recon**. Everything else in the dataset is not used (see
`data_and_model_questions.md`, "What did we remove").

**Attacker IPs.** In our own recordings, rows from these addresses are the attack; all other senders are normal.

In [ ]:
FEATURES = ["Protocol Type", "Tot size", "Min", "Max", "AVG", "Std",
            "syn_flag_number", "ack_flag_number", "psh_flag_number", "rst_flag_number", "fin_flag_number",
            "TCP", "UDP", "ICMP"]
CLASSES = ["Benign", "Flood", "Recon"]
FLOODS = {"DDoS-SYN_Flood", "DDoS-UDP_Flood", "DDoS-ICMP_Flood", "DDoS-TCP_Flood", "DDoS-PSHACK_Flood",
          "DDoS-RSTFINFlood", "DoS-SYN_Flood", "DoS-UDP_Flood", "DoS-TCP_Flood"}
RECON = {"Recon-PortScan", "Recon-OSScan", "VulnerabilityScan"}
ATTACKER_IPS = {"172.28.60.66",       # the attacker container in the Docker testbed
                "172.29.2.52"}        # the laptop in the real lab (17 Nov)
MAX_BYTES = 1_000_000                 # the model file must stay < 1 MB (project target)
THRESHOLD = 0.70                      # same as the live detector: P(attack) below 0.70 -> Benign


def dataset_class(label):
    if label == "BenignTraffic": return "Benign"
    if label in FLOODS: return "Flood"
    if label in RECON: return "Recon"
    return None                       # every other attack: not used

## 2. Dataset rows (CICIoT2023)

From each split we keep only Benign / Flood / Recon rows. For training and validation we take a **random 20% of every
attack type** (stratified: the mix stays the same). 20% is already ~900,000 rows; early stopping makes more rows
mostly slower, not better. The test split is used **100%**.

In [ ]:
def load_dataset(split, fraction, seed):
    df = pd.read_parquet(CACHE / f"{split}.parquet", columns=FEATURES + ["label"])
    df["y"] = df["label"].map(dataset_class)
    df = df[df["y"].notna()]
    if fraction < 1:
        df = df.groupby("label", observed=True, group_keys=False).sample(frac=fraction, random_state=seed)
    df = df.reset_index(drop=True)
    df["source"] = "CICIoT2023"
    df["label"] = df["label"].astype(str)                 # the original attack name, e.g. DDoS-SYN_Flood
    return df


ds_train = load_dataset("train", 0.2, 0)
ds_val = load_dataset("validation", 0.2, 1)
ds_test = load_dataset("test", 1.0, 2)
for name, d in [("train", ds_train), ("validation", ds_val), ("test", ds_test)]:
    print(f"{name:<11}{len(d):>9,} rows ", d["y"].value_counts().to_dict())

## 3. Our own recorded rows

Recorded with `pi/record.py` (the same feature code as the live detector), one csv per recording in
`training/data/`. Labels come from the file name and the sender:
- `benign*.csv`: everything Benign (our NodeMCU sensors sending MQTT)
- `flood_*.csv` / `recon_*.csv`: rows from the attacker IP = Flood / Recon, all other senders = Benign
- `flood_bots_*.csv`: recorded only while the NodeMCU bots flooded, so every sender is a bot = Flood
- real lab files (`*_lab*`): the laptop's slow windows (SSH etc., < 20 packets/s) are dropped

They are split **70 / 15 / 15** at random (seed 0) into train / validation / test.

In [ ]:
parts = []
for f in sorted(OUR_DATA.glob("*.csv")):
    d = pd.read_csv(f)
    if "_lab" in f.stem and "pps" in d:
        d = d[~(d["src"].isin(ATTACKER_IPS) & (d["pps"] < 20))]
    kind = "Flood" if f.name.startswith("flood_") else "Recon" if f.name.startswith("recon_") else "Benign"
    if f.name.startswith("flood_bots"):
        d = d[~d["src"].isin(ATTACKER_IPS)].copy()
        d["y"] = "Flood"
    else:
        d["y"] = np.where(d["src"].isin(ATTACKER_IPS), kind, "Benign")
    d["label"] = f.stem                                   # the recording it came from
    parts.append(d)
ours_all = pd.concat(parts, ignore_index=True)
ours_all["source"] = "ours"
split = np.random.default_rng(0).choice(["tr", "va", "te"], size=len(ours_all), p=[0.7, 0.15, 0.15])
ours = {k: ours_all[split == k].reset_index(drop=True) for k in ("tr", "va", "te")}
print(f"our rows: {len(ours_all):,}", ours_all["y"].value_counts().to_dict())
for k, d in ours.items():
    print(f"  {k}: {len(d):,}")

## 4. Train / validation / test

- **train** = 20% dataset sample + 70% of our rows (the model learns from these)
- **validation** = 20% of the dataset's validation split + 15% of ours (only used to decide when to stop adding trees)
- **test** = the dataset's full test split, and separately the other 15% of ours (never seen in training)

In [ ]:
KEEP = FEATURES + ["y", "label", "source"]
train = pd.concat([ds_train[KEEP], ours["tr"][KEEP]], ignore_index=True)
val = pd.concat([ds_val[KEEP], ours["va"][KEEP]], ignore_index=True)
test_dataset, test_ours = ds_test[KEEP], ours["te"][KEEP]
for name, d in [("train", train), ("validation", val), ("test (dataset)", test_dataset), ("test (ours)", test_ours)]:
    print(f"{name:<15}{len(d):>9,} rows ", d["y"].value_counts().to_dict())

## 5. Train LightGBM and rank the features

**LightGBM** = many small decision trees, each one correcting the previous ones (gradient boosting). Settings:
- `num_leaves 15`: small trees (keeps the file small and fast on the Pi)
- `max_bin 63`: each feature is cut into at most 63 value ranges
- `learning_rate 0.1`, up to 300 trees, **early stopping**: stop when the validation score has not improved for
  20 trees
- **class weights**: Recon is ~0.6% of the rows, so each Recon row counts more (square root of the inverse share),
  otherwise the model would just answer "Flood"
- **size cap**: if the file is >= 1 MB, drop trees from the end until it fits (this is our "pruning")

There is no INT8 quantization: a tree model stores split thresholds, not big weight matrices, and it is already
< 1 MB and ~1 ms on the Pi.

**SHAP** (mean |SHAP value|) says how much each feature moved the predictions on average. We rank the 14 features
with it; the final model is trained with the features in this ranked order.

In [ ]:
def file_size(booster, rounds):
    return len(booster.model_to_string(num_iteration=rounds).encode())


def train_model(features):
    data = {"tr": train, "va": val}
    X = {k: d[features].to_numpy(np.float32) for k, d in data.items()}
    y = {k: d["y"].map(CLASSES.index).to_numpy() for k, d in data.items()}
    count = np.bincount(y["tr"], minlength=3)
    w = (len(y["tr"]) / (3 * count)) ** 0.5               # class weights
    w = w / w[y["tr"]].mean()
    params = {"objective": "multiclass", "num_class": 3, "learning_rate": 0.1, "num_leaves": 15, "max_bin": 63,
              "bagging_fraction": 0.8, "bagging_freq": 1, "feature_fraction": 0.8, "verbose": -1, "seed": 0}
    booster = lgb.train(params, lgb.Dataset(X["tr"], y["tr"], weight=w[y["tr"]]), num_boost_round=300,
                        valid_sets=[lgb.Dataset(X["va"], y["va"])],
                        callbacks=[lgb.early_stopping(20, verbose=False)])
    rounds = booster.best_iteration or booster.current_iteration()
    while rounds > 1 and file_size(booster, rounds) >= MAX_BYTES:     # size cap (< 1 MB)
        rounds -= 1
    return booster, rounds


t0 = time.time()
first, first_rounds = train_model(FEATURES)
rows = val.sample(min(20000, len(val)), random_state=0)[FEATURES].to_numpy(np.float32)
shap = np.abs(first.predict(rows, pred_contrib=True, num_iteration=first_rounds).reshape(len(rows), 3, -1))
shap = shap[:, :, :-1].sum(1).mean(0)                    # mean |SHAP| per feature, summed over the 3 classes
ranking = sorted(FEATURES, key=lambda f: -shap[FEATURES.index(f)])
for i, f in enumerate(ranking):
    print(f"{i + 1:2d}. {f:<16} mean |SHAP| {shap[FEATURES.index(f)]:.3f}")
print(f"({time.time() - t0:.0f} s)")

## 6. The final model (= `top14`, the one on the Pi)

Same 14 features, in ranked order (the order matters only for how the file is written; the Pi reads the order from
`meta.json`).

In [ ]:
model, rounds = train_model(ranking)
print("trees per class:", rounds, "| size:", round(file_size(model, rounds) / 1024, 1), "KB")

## 7. Scores on the test rows

The decision rule is the one used live: **P(attack) = 1 - P(Benign)**; if it is below 0.70 the window is Benign,
otherwise the most likely attack class (Flood or Recon).

- **precision**: when the model says "Flood", how often it is really a flood
- **recall**: of all real floods, how many it caught
- **F1**: precision and recall combined into one number (1.0 = perfect)
- **macro avg F1**: the average of the 3 classes' F1, each class counting the same (the small Recon class pulls it down)

In [ ]:
def decide(booster, X):
    p = booster.predict(X, num_iteration=rounds)
    return np.where(1 - p[:, 0] < THRESHOLD, 0, 1 + p[:, 1:].argmax(1))


for name, d in [("DATASET test rows", test_dataset), ("OUR test rows", test_ours)]:
    y = d["y"].map(CLASSES.index).to_numpy()
    pred = decide(model, d[ranking].to_numpy(np.float32))
    print(f"===== {name} ({len(d):,}) =====")
    print(classification_report(y, pred, labels=[0, 1, 2], target_names=CLASSES, digits=4, zero_division=0))
    print("confusion matrix (rows = truth, columns = prediction):")
    print(pd.DataFrame(confusion_matrix(y, pred, labels=[0, 1, 2]),
                       index=["true " + c for c in CLASSES], columns=["pred " + c for c in CLASSES]), "\n")

## 8. Save, and check it is the same model as on the Pi

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
model.save_model(str(OUT / "model.txt"), num_iteration=rounds)
json.dump({"features": ranking, "model": {"file": "model.txt", "classes": CLASSES, "rounds": rounds}},
          open(OUT / "meta.json", "w"), indent=2)


def trees_only(path):
    # the model file without its last lines (LightGBM writes the training parameters there)
    text = Path(path).read_text()
    return text.split("end of trees")[0]


same = trees_only(OUT / "model.txt") == trees_only(DEPLOYED / "model.txt")
print("saved to", OUT)
print("identical trees to the deployed pi/models/top14/model.txt:", same)

## 9. Export the exact rows used (for anyone who wants to look at the data)

Written to `evaluation_matrices/dataset/`. Columns: the 14 features, `y` (the class the model learned),
`label` (the original dataset attack name, or for our rows the recording file), `source` (CICIoT2023 or ours).

In [ ]:
DATASET_OUT.mkdir(parents=True, exist_ok=True)
for name, d in [("train", train), ("validation", val), ("test_dataset", test_dataset), ("test_ours", test_ours)]:
    d.to_csv(DATASET_OUT / f"{name}.csv", index=False)
    print(f"{name}.csv  {len(d):,} rows  {(DATASET_OUT / f'{name}.csv').stat().st_size / 1e6:.0f} MB")